In [ ]:
from pathlib import Path
import sqlparse

def construct_query(sql_file, placeholders):
    # Read the query file
    project_root = get_project_root_folder()
    query_file_path = Path(f"{project_root}/sql/reports/{sql_file}").absolute()
    with open(query_file_path.absolute(), "r") as f:
        query = f.read().replace("\n", " ")

    # Replace all placeholders
    for key, value in placeholders.items():
        query = query.replace(f"${key}", str(value).replace("'", "''"))

    # Format the query
    query = sqlparse.format(query, reindent=True, keyword_case='upper')

    return query

In [ ]:
import pandas as pd
import sqlite3
import os

def query_data(query):
    # Connect to the database, execute the query and read the results into a dataframe
    database_path = os.environ["DAILY_TASK_MANAGER_DB"]
    with sqlite3.connect(database_path) as connection:
        df = pd.read_sql_query(query, connection)

    # Parse consistently named report date columns when they are present.
    for column in df.columns:
        if column == "Date" or column.endswith(" Date"):
            df[column] = pd.to_datetime(df[column], errors="coerce")
    return df